# CLV Prediction

In [2]:
# Model 3: predict each customer's realized lifetime value with regression.
# CLV is successful payment revenue through churn date, or the last payment date for non-churned customers.
# Predictors are frozen before 2025-10-02, and payment amounts are deliberately excluded from the predictors.

# Import Path so data folders can be found reliably from the notebook.  # Explain file paths.
from pathlib import Path
# Import find_spec to check whether optional XGBoost is installed.  # Explain optional dependency check.
from importlib.util import find_spec
# Import Matplotlib for a basic actual-versus-predicted diagnostic chart.  # Explain plotting.
import matplotlib.pyplot as plt
# Import NumPy for numeric calculations.  # Explain numeric helpers.
import numpy as np
# Import pandas to load and combine customer tables.  # Explain tabular data.
import pandas as pd
# Import the preprocessing tools used inside each model's pipeline.  # Explain preparation tools.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Import the three requested regression model types, plus metrics and validation tools.  # Explain model comparison.
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fix random behavior so the split, cross-validation, and tree models are reproducible.  # Explain reproducibility.
RANDOM_STATE = 42
# Define the date before which every predictor must be available.  # Explain prediction snapshot.
FEATURE_CUTOFF = pd.Timestamp("2025-10-02")
# Start from the folder where the notebook was launched.  # Explain starting folder.
PROJECT_DIR = Path.cwd()
# Move one folder up when the notebook was launched inside the notebooks folder.  # Explain path fallback.
if not (PROJECT_DIR / "data" / "processed").exists():
    PROJECT_DIR = PROJECT_DIR.parent
# Point to the project's cleaned customer tables.  # Explain source folder.
DATA_DIR = PROJECT_DIR / "data" / "processed"
# Point to the separate file that will contain held-out predictions.  # Explain output path.
OUTPUT_FILE = DATA_DIR / "output_clv_test_predictions.csv"

# Load customer attributes and parse registration dates.  # Explain customer source.
customers = pd.read_csv(DATA_DIR / "customers.csv", parse_dates=["registration_date"])
# Load churn dates only to define the end of each customer's target-value observation period.  # Explain target boundary source.
churn_labels = pd.read_csv(DATA_DIR / "churn_labels.csv", parse_dates=["churn_date"])
# Load payment dates and amounts to construct the target, not model inputs.  # Explain target source.
payments = pd.read_csv(DATA_DIR / "payments.csv", parse_dates=["payment_date"])
# Read viewing records and dates so only pre-cutoff activity contributes features.  # Explain engagement source.
viewing = pd.read_csv(DATA_DIR / "viewing_activity.csv", parse_dates=["viewing_date"])
# Read subscription records and start dates for pre-cutoff plan features.  # Explain billing source.
subscriptions = pd.read_csv(DATA_DIR / "subscriptions.csv", parse_dates=["subscription_start_date"])
# Read dated customer feedback for pre-cutoff satisfaction features.  # Explain satisfaction source.
feedback = pd.read_csv(DATA_DIR / "customer_feedback.csv", parse_dates=["feedback_date"])
# Find the last date available in the payment table for customers without a churn date.  # Explain observation end.
PAYMENT_DATA_END = payments["payment_date"].max()
# Print both dates so the feature and target periods are clear to the reader.  # Explain date report.
print(f"Feature snapshot cutoff: {FEATURE_CUTOFF.date()}")
print(f"Latest payment date in the data: {PAYMENT_DATA_END.date()}")

# Join churn dates to payments so each payment can be checked against that customer's observation end.  # Explain target construction.
payments_with_end_date = payments.merge(churn_labels[["customer_id", "churn_date"]], on="customer_id", how="left", validate="many_to_one")
# Use churn date for churned customers and the last observed payment date for others.  # Explain target horizon.
payments_with_end_date["value_observation_end"] = payments_with_end_date["churn_date"].fillna(PAYMENT_DATA_END)
# Keep successful payments no later than each customer's observation end.  # Explain SQL-compatible target rule.
successful_payments = payments_with_end_date.loc[
    payments_with_end_date["payment_status"].str.casefold().eq("success")
    & payments_with_end_date["payment_date"].le(payments_with_end_date["value_observation_end"])
]
# Sum the retained successful amounts into the exact regression target column.  # Explain target aggregation.
clv_target = successful_payments.groupby("customer_id").agg(
    customer_lifetime_value=("amount", "sum")
).reset_index()

# Keep customers who were registered by the feature snapshot date.  # Explain eligible cohort.
customer_features = customers.loc[customers["registration_date"].le(FEATURE_CUTOFF), [
    "customer_id", "age", "gender", "country", "registration_date",
    "acquisition_channel", "preferred_language",
]].copy()
# Calculate tenure using only dates known at the feature snapshot.  # Explain tenure feature.
customer_features["tenure_days"] = (FEATURE_CUTOFF - customer_features["registration_date"]).dt.days
# Remove the raw registration date because tenure captures its usable time information.  # Explain date exclusion.
customer_features = customer_features.drop(columns="registration_date")
# Keep only viewing events that occurred by the feature cutoff.  # Explain engagement cutoff.
viewing = viewing.loc[viewing["viewing_date"].le(FEATURE_CUTOFF)]
# Aggregate engagement behavior into one row per customer.  # Explain engagement features.
viewing_features = viewing.groupby("customer_id").agg(
    viewing_sessions=("viewing_id", "nunique"),
    watch_minutes=("watch_duration_minutes", "sum"),
    mean_completion_percentage=("completion_percentage", "mean"),
    distinct_devices=("device_type", "nunique"),
).reset_index()
# Keep subscription records whose start date was known by the feature cutoff.  # Explain subscription cutoff.
subscriptions = subscriptions.loc[subscriptions["subscription_start_date"].le(FEATURE_CUTOFF)].copy()
# Sort subscriptions so the final plan row for each customer is their latest pre-cutoff plan.  # Explain latest plan.
subscriptions = subscriptions.sort_values(["customer_id", "subscription_start_date", "subscription_id"])
# Aggregate plan price, plan identifier, and number of prior subscriptions.  # Explain billing features.
subscription_features = subscriptions.groupby("customer_id").agg(
    latest_plan_id=("plan_id", "last"),
    latest_monthly_price=("monthly_price", "last"),
    subscription_count=("subscription_id", "nunique"),
).reset_index()
# Keep only feedback received by the feature cutoff.  # Explain satisfaction cutoff.
feedback = feedback.loc[feedback["feedback_date"].le(FEATURE_CUTOFF)].copy()
# Mark positive sentiment with 1 so its average is the positive-feedback share.  # Explain sentiment encoding.
feedback["positive_feedback"] = feedback["sentiment_label"].str.casefold().eq("positive").astype(int)
# Aggregate satisfaction ratings, positive share, and feedback count.  # Explain satisfaction features.
feedback_features = feedback.groupby("customer_id").agg(
    mean_feedback_rating=("rating", "mean"),
    positive_feedback_share=("positive_feedback", "mean"),
    feedback_count=("feedback_id", "nunique"),
).reset_index()
# Start from eligible customers and attach their pre-cutoff behavior summaries.  # Explain feature joining.
model_data = customer_features.merge(viewing_features, on="customer_id", how="left", validate="one_to_one")
# Attach plan features without introducing post-cutoff subscriptions.  # Explain subscription join.
model_data = model_data.merge(subscription_features, on="customer_id", how="left", validate="one_to_one")
# Attach historical satisfaction features.  # Explain feedback join.
model_data = model_data.merge(feedback_features, on="customer_id", how="left", validate="one_to_one")
# Attach the target separately; it is never included in the predictor columns.  # Explain target join.
model_data = model_data.merge(clv_target, on="customer_id", how="left", validate="one_to_one")
# Treat missing activity counts as zero because no matching record means no recorded event.  # Explain count defaults.
count_columns = ["viewing_sessions", "distinct_devices", "subscription_count", "feedback_count"]
model_data[count_columns] = model_data[count_columns].fillna(0)
# Treat no recorded watch time as zero minutes.  # Explain watch-time default.
model_data["watch_minutes"] = model_data["watch_minutes"].fillna(0)
# Treat customers without a successful payment as zero observed lifetime value.  # Explain target default.
model_data["customer_lifetime_value"] = model_data["customer_lifetime_value"].fillna(0)
# Show the cohort size and target range before modeling.  # Explain dataset summary.
print(f"Eligible customers: {len(model_data):,}")
print(f"CLV target range: ${model_data['customer_lifetime_value'].min():.2f} to ${model_data['customer_lifetime_value'].max():.2f}")

# Remove customer IDs and the target from the predictor table.  # Explain feature-target separation.
X = model_data.drop(columns=["customer_id", "customer_lifetime_value"])
# Keep the target series under the requested name.  # Explain regression target.
y = model_data["customer_lifetime_value"]
# Reserve 20 percent of customers for one final evaluation.  # Explain test split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
# Detect numeric columns from training data only.  # Explain numeric column detection.
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
# Detect categorical columns from training data only.  # Explain categorical column detection.
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()
# Impute numeric gaps and scale numeric values inside every training fold.  # Explain numeric preparation.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
# Impute missing categories and one-hot encode categories, including unseen test values safely.  # Explain category preparation.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
# Apply the appropriate preparation to each column type and discard no other columns.  # Explain preprocessing.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")

# Define the three requested regression candidates.  # Explain candidate models.
candidates = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=150, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1
    ),
}
# Add XGBoost when its package is installed, while keeping the other candidates runnable without it.  # Explain optional model.
if find_spec("xgboost") is not None:
    from xgboost import XGBRegressor
    candidates["XGBoost Regressor"] = XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=RANDOM_STATE,
        n_jobs=1, verbosity=0,
    )
else:
    print("XGBoost Regressor skipped because the xgboost package is not installed.")
# Use the same shuffled five folds to compare candidate models fairly.  # Explain cross-validation.
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# Score average absolute error, root mean squared error, and explained variance in each fold.  # Explain CV metrics.
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}
# Collect mean cross-validation metrics and retain each model pipeline.  # Explain result storage.
cv_rows, candidate_pipelines = [], {}
# Fit and score every model using training data only.  # Explain model comparison loop.
for model_name, estimator in candidates.items():
    # Keep preprocessing attached so each CV fold learns its own imputation, scaling, and categories.  # Explain pipeline.
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", estimator),
    ])
    # Run five-fold cross-validation and fail clearly if a candidate cannot fit.  # Explain CV execution.
    cv_scores = cross_validate(
        model_pipeline, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=1, error_score="raise",
    )
    # Save this pipeline for final training after model selection.  # Explain pipeline storage.
    candidate_pipelines[model_name] = model_pipeline
    # Convert negative error scores back to positive dollar errors and average across folds.  # Explain score signs.
    cv_rows.append({
        "model": model_name,
        "cv_mae": -cv_scores["test_mae"].mean(),
        "cv_rmse": -cv_scores["test_rmse"].mean(),
        "cv_r2": cv_scores["test_r2"].mean(),
    })
# Rank models by lowest cross-validated MAE, then lowest RMSE as a tie-breaker.  # Explain selection rule.
cv_results = pd.DataFrame(cv_rows).sort_values(["cv_mae", "cv_rmse"])
# Print the scores used to select the final regressor.  # Explain CV report.
print("\nFive-fold cross-validation on training data:")
print(cv_results.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
# Select the candidate with the smallest average absolute error.  # Explain final model selection.
selected_name = cv_results.iloc[0]["model"]
selected_model = candidate_pipelines[selected_name]
print(f"\nSelected model by cross-validated MAE: {selected_name}")
# Fit the selected pipeline on the full training split.  # Explain final fitting.
selected_model.fit(X_train, y_train)
# Predict lifetime value for customers not used in fitting.  # Explain held-out prediction.
test_predictions = selected_model.predict(X_test)
# Calculate the three requested performance metrics in the target's monetary units.  # Explain final metrics.
test_metrics = {
    "MAE": mean_absolute_error(y_test, test_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
    "R2": r2_score(y_test, test_predictions),
}
# Print the final held-out results.  # Explain test report.
print("\nHeld-out test metrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name:>8}: {metric_value:.3f}")
# Compare average and predicted values to make prediction bias easier to spot.  # Explain prediction summary.
print(f"\nMean actual CLV: ${y_test.mean():.2f}; mean predicted CLV: ${test_predictions.mean():.2f}")
# Explain what the selected model's score and error mean in this customer-value context.  # Explain model justification.
selected_cv = cv_results.iloc[0]
print("\nModel-choice justification:")
print(
    f"{selected_name} was selected because it had the lowest five-fold training "
    f"MAE (${selected_cv['cv_mae']:.2f}); its CV RMSE was ${selected_cv['cv_rmse']:.2f} "
    f"and CV R2 was {selected_cv['cv_r2']:.3f}. On the untouched test split, "
    f"MAE was ${test_metrics['MAE']:.2f}, RMSE was ${test_metrics['RMSE']:.2f}, "
    f"and R2 was {test_metrics['R2']:.3f}. MAE is the typical absolute dollar error; "
    "RMSE penalizes large misses more strongly; R2 summarizes how much target variation "
    "the model explains relative to predicting the test-set mean. These scores describe "
    "historical realized value through the available data, not guaranteed future revenue."
)
# Save held-out actual and predicted values for review without exporting training predictions.  # Explain prediction export.
prediction_report = pd.DataFrame({
    "customer_id": model_data.loc[X_test.index, "customer_id"].to_numpy(),
    "actual_customer_lifetime_value": y_test.to_numpy(),
    "predicted_customer_lifetime_value": test_predictions,
})
prediction_report.to_csv(OUTPUT_FILE, index=False)
# Confirm where the test predictions were saved.  # Explain saved result.
print(f"\nSaved held-out predictions to: {OUTPUT_FILE}")

Feature snapshot cutoff: 2025-10-02
Latest payment date in the data: 2026-05-31
Eligible customers: 7,277
CLV target range: $20.97 to $355.35

Five-fold cross-validation on training data:
                  model  cv_mae  cv_rmse  cv_r2
      XGBoost Regressor  12.477   17.929  0.959
Random Forest Regressor  12.638   18.209  0.958
      Linear Regression  15.848   23.577  0.929

Selected model by cross-validated MAE: XGBoost Regressor

Held-out test metrics:
     MAE: 12.210
    RMSE: 17.124
      R2: 0.962

Mean actual CLV: $186.30; mean predicted CLV: $185.66

Model-choice justification:
XGBoost Regressor was selected because it had the lowest five-fold training MAE ($12.48); its CV RMSE was $17.93 and CV R2 was 0.959. On the untouched test split, MAE was $12.21, RMSE was $17.12, and R2 was 0.962. MAE is the typical absolute dollar error; RMSE penalizes large misses more strongly; R2 summarizes how much target variation the model explains relative to predicting the test-set mean. Thes